# Notebook 3: Large-Scale Tweedie Framework Comparison

## PyTorch vs. PySpark vs. Dask — Head-to-Head Benchmark

This notebook provides a **controlled comparison** of three scalable frameworks for Tweedie distribution-based regression on a **single large synthetic dataset** (10M rows, 80% zeros, Gamma-distributed positives).

### What We Measure
| Metric | Description |
|--------|-------------|
| **Training Time** | Wall-clock time from data loading to fitted model |
| **Peak Memory** | Maximum resident memory during training |
| **Prediction Throughput** | Rows predicted per second |
| **Convergence** | Loss trajectory over training iterations |
| **Predictive Accuracy** | MAE, RMSE, Tweedie Deviance on held-out test set |

### The Tweedie Loss Function

For power parameter $p \in (1, 2)$, the Tweedie deviance for observation $(y_i, \mu_i)$ is:

$$d(y_i, \mu_i) = 2 \left[ \frac{y_i^{2-p}}{(1-p)(2-p)} - \frac{y_i \mu_i^{1-p}}{1-p} + \frac{\mu_i^{2-p}}{2-p} \right]$$

This loss naturally handles zero-inflated, right-skewed targets via its compound Poisson-Gamma interpretation.

---

## 1. Environment Setup

In [ ]:
# !pip install torch pyspark dask dask-ml scikit-learn xgboost lightgbm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import tracemalloc
import gc
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_tweedie_deviance
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import TweedieRegressor

# Benchmark results collector
benchmarks = {}

print("Environment ready!")
print(f"NumPy: {np.__version__}")

## 2. Synthetic Data Generation

We generate a **10M-row dataset** with:
- 80% exact zeros (no-event observations)
- 20% positive values drawn from a Gamma distribution (right-skewed)
- 15 features (mix of continuous and categorical)

This mimics insurance claims, healthcare costs, or sparse retail sales.

In [ ]:
def generate_tweedie_data(n_rows, p_power=1.5, phi=2.0, zero_frac=0.8,
                         n_features=15, seed=42):
    """Generate synthetic Tweedie-distributed data.

    Parameters:
    -----------
    n_rows : int - number of observations
    p_power : float - Tweedie power parameter (1 < p < 2)
    phi : float - dispersion parameter
    zero_frac : float - approximate fraction of zeros
    n_features : int - number of predictor features
    seed : int - random seed

    Returns:
    --------
    X : ndarray (n_rows, n_features) - feature matrix
    y : ndarray (n_rows,) - target values (Tweedie-distributed)
    """
    rng = np.random.RandomState(seed)

    # Generate features
    X = np.zeros((n_rows, n_features), dtype=np.float32)
    X[:, 0] = rng.uniform(18, 80, n_rows)           # age-like
    X[:, 1] = rng.binomial(1, 0.5, n_rows)           # binary
    X[:, 2] = np.clip(rng.normal(28, 6, n_rows), 15, 55)  # bmi-like
    X[:, 3] = rng.binomial(1, 0.15, n_rows)           # risk factor
    X[:, 4] = rng.poisson(1.5, n_rows)                # count feature
    X[:, 5] = rng.randint(0, 10, n_rows)               # category
    X[:, 6] = rng.uniform(0, 1, n_rows)                # continuous
    X[:, 7] = rng.normal(0, 1, n_rows)                 # standard normal
    X[:, 8] = rng.exponential(1, n_rows)                # skewed feature
    X[:, 9] = np.sin(2 * np.pi * rng.uniform(0, 1, n_rows))  # cyclic
    X[:, 10] = rng.uniform(10, 500, n_rows)             # price-like
    X[:, 11] = X[:, 0] ** 2 / 1000                      # quadratic
    X[:, 12] = X[:, 2] * X[:, 3]                        # interaction
    X[:, 13] = np.log1p(X[:, 10])                       # log transform
    X[:, 14] = rng.choice([0,1,2,3], n_rows)            # ordinal

    # Linear predictor
    beta = rng.uniform(-0.5, 0.5, n_features)
    beta[0] = 0.02   # age effect
    beta[3] = 0.8    # risk factor effect
    beta[4] = 0.3    # conditions effect
    log_mu = 2.0 + X @ beta
    mu = np.exp(np.clip(log_mu, -5, 10))

    # Generate Tweedie (compound Poisson-Gamma)
    lam = mu**(2 - p_power) / (phi * (2 - p_power))
    alpha = (2 - p_power) / (p_power - 1)
    beta_g = phi * (p_power - 1) * mu**(p_power - 1)

    N = rng.poisson(lam)
    y = np.zeros(n_rows, dtype=np.float32)
    for i in range(n_rows):
        if N[i] > 0:
            y[i] = rng.gamma(alpha, beta_g[i], N[i]).sum()

    actual_zero_frac = (y == 0).mean()
    print(f"Generated {n_rows:,} rows | {actual_zero_frac*100:.1f}% zeros | "
          f"Non-zero mean: {y[y>0].mean():.2f} | Max: {y.max():.2f}")

    return X, y


# Generate the benchmark dataset
print("=== Generating 10M-row Benchmark Dataset ===")
print("(This may take a few minutes...)\n")

t0 = time.time()
N_TOTAL = 10_000_000
X_all, y_all = generate_tweedie_data(N_TOTAL, p_power=1.5, phi=2.0, seed=42)

# Train/test split (80/20)
split = int(0.8 * N_TOTAL)
X_train_all, X_test_all = X_all[:split], X_all[split:]
y_train_all, y_test_all = y_all[:split], y_all[split:]

gen_time = time.time() - t0
print(f"\nData generation time: {gen_time:.1f}s")
print(f"Training: {X_train_all.shape[0]:,} | Test: {X_test_all.shape[0]:,}")
print(f"Memory: X={X_all.nbytes/1e9:.2f} GB, y={y_all.nbytes/1e6:.0f} MB")

## 3. Baseline: Scikit-learn TweedieRegressor (Single-Core)

Establishes the baseline performance on a subsample (scikit-learn cannot handle 10M rows efficiently).

In [ ]:
# Scikit-learn baseline (subsample for feasibility)
SUB_SIZE = 500_000
sub_idx = np.random.choice(split, SUB_SIZE, replace=False)
X_sub = X_train_all[sub_idx]
y_sub = y_train_all[sub_idx]

scaler_sk = StandardScaler()
X_sub_s = scaler_sk.fit_transform(X_sub)
X_test_s = scaler_sk.transform(X_test_all)

tracemalloc.start()
t0 = time.time()

sk_model = TweedieRegressor(power=1.5, alpha=0.1, link='log', max_iter=500)
sk_model.fit(X_sub_s, y_sub)

time_sk = time.time() - t0
_, mem_sk = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Predict
t_pred = time.time()
y_pred_sk = np.clip(sk_model.predict(X_test_s), 0, None)
pred_time_sk = time.time() - t_pred

mae_sk = mean_absolute_error(y_test_all, y_pred_sk)
rmse_sk = np.sqrt(mean_squared_error(y_test_all, y_pred_sk))
td_sk = mean_tweedie_deviance(y_test_all, np.clip(y_pred_sk, 1e-6, None), power=1.5)
throughput_sk = len(y_test_all) / pred_time_sk

benchmarks['sklearn'] = {
    'Framework': 'Scikit-learn GLM',
    'Data Used': f'{SUB_SIZE:,} (subsample)',
    'Train Time (s)': round(time_sk, 2),
    'Peak Memory (MB)': round(mem_sk / 1e6, 1),
    'Pred Time (s)': round(pred_time_sk, 2),
    'Throughput (rows/s)': f'{throughput_sk:,.0f}',
    'MAE': round(mae_sk, 2),
    'RMSE': round(rmse_sk, 2),
    'Tweedie Dev': round(td_sk, 4)
}

print(f"Scikit-learn Baseline (trained on {SUB_SIZE:,} rows)")
print(f"  Train: {time_sk:.2f}s | Memory: {mem_sk/1e6:.1f} MB")
print(f"  MAE: {mae_sk:.2f} | RMSE: {rmse_sk:.2f} | TD: {td_sk:.4f}")
print(f"  Prediction throughput: {throughput_sk:,.0f} rows/s")

## 4. PyTorch: Custom Tweedie Loss with Mini-Batch Training

PyTorch handles the full 8M training rows via **mini-batch gradient descent** with a custom Tweedie negative log-likelihood loss.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


class TweedieLoss(nn.Module):
    """Tweedie negative log-likelihood loss.

    For p in (1,2), the loss per sample is:
      L = -y * mu^(1-p)/(1-p) + mu^(2-p)/(2-p)
    where mu = exp(raw_output) ensures positivity.
    """
    def __init__(self, p=1.5):
        super().__init__()
        self.p = p

    def forward(self, log_mu, y_true):
        mu = torch.exp(torch.clamp(log_mu, -10, 10))
        term1 = -y_true * torch.pow(mu + 1e-8, 1 - self.p) / (1 - self.p)
        term2 = torch.pow(mu + 1e-8, 2 - self.p) / (2 - self.p)
        return torch.mean(term1 + term2)


class TweedieNet(nn.Module):
    """Deep network for Tweedie regression."""
    def __init__(self, input_dim, hidden=[512, 256, 128, 64]):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden:
            layers.extend([
                nn.Linear(prev, h), nn.LeakyReLU(0.1),
                nn.BatchNorm1d(h), nn.Dropout(0.2)
            ])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# Prepare tensors
print("Preparing PyTorch data...")
scaler_pt = StandardScaler()
X_train_pt = torch.tensor(scaler_pt.fit_transform(X_train_all), dtype=torch.float32)
X_test_pt = torch.tensor(scaler_pt.transform(X_test_all), dtype=torch.float32)
y_train_pt = torch.tensor(y_train_all, dtype=torch.float32)
y_test_pt = torch.tensor(y_test_all, dtype=torch.float32)

train_ds = TensorDataset(X_train_pt, y_train_pt)
train_loader = DataLoader(train_ds, batch_size=16384, shuffle=True, num_workers=0, pin_memory=True)

print(f"Training batches: {len(train_loader)} x 16,384 = {len(train_ds):,}")

In [ ]:
# Train PyTorch model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

model_pt = TweedieNet(input_dim=15).to(device)
criterion = TweedieLoss(p=1.5)
optimizer = torch.optim.Adam(model_pt.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

tracemalloc.start()
t0 = time.time()

losses_pt = []
for epoch in range(10):
    model_pt.train()
    epoch_loss = 0
    n_b = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model_pt(xb)
        loss = criterion(out, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pt.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_b += 1
    scheduler.step()
    avg = epoch_loss / n_b
    losses_pt.append(avg)
    print(f"  Epoch {epoch+1:2d}/10 | Loss: {avg:.6f}")

time_pt = time.time() - t0
_, mem_pt = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Predict
model_pt.eval()
t_pred = time.time()
with torch.no_grad():
    # Predict in chunks to manage memory
    preds = []
    for i in range(0, len(X_test_pt), 100_000):
        chunk = X_test_pt[i:i+100_000].to(device)
        preds.append(torch.exp(model_pt(chunk)).cpu().numpy())
    y_pred_pt = np.clip(np.concatenate(preds), 0, None)
pred_time_pt = time.time() - t_pred

mae_pt = mean_absolute_error(y_test_all, y_pred_pt)
rmse_pt = np.sqrt(mean_squared_error(y_test_all, y_pred_pt))
td_pt = mean_tweedie_deviance(y_test_all, np.clip(y_pred_pt, 1e-6, None), power=1.5)
throughput_pt = len(y_test_all) / pred_time_pt

benchmarks['pytorch'] = {
    'Framework': 'PyTorch NN',
    'Data Used': f'{split:,} (full)',
    'Train Time (s)': round(time_pt, 2),
    'Peak Memory (MB)': round(mem_pt / 1e6, 1),
    'Pred Time (s)': round(pred_time_pt, 2),
    'Throughput (rows/s)': f'{throughput_pt:,.0f}',
    'MAE': round(mae_pt, 2),
    'RMSE': round(rmse_pt, 2),
    'Tweedie Dev': round(td_pt, 4)
}

print(f"\nPyTorch Complete!")
print(f"  Train: {time_pt:.1f}s | Memory: {mem_pt/1e6:.1f} MB | Throughput: {throughput_pt:,.0f} rows/s")
print(f"  MAE: {mae_pt:.2f} | RMSE: {rmse_pt:.2f} | TD: {td_pt:.4f}")

# Cleanup GPU memory
del X_train_pt, y_train_pt, train_ds, train_loader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. PySpark: Distributed Tweedie GLM

PySpark distributes the computation across cores/nodes using its native `GeneralizedLinearRegression` with `family="tweedie"`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GeneralizedLinearRegression

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("TweedieBenchmark") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Convert to Spark DataFrame
feat_names = [f'f{i}' for i in range(15)]
print("Loading data into Spark...")

tracemalloc.start()
t0 = time.time()

train_pdf = pd.DataFrame(X_train_all, columns=feat_names)
train_pdf['target'] = y_train_all
test_pdf = pd.DataFrame(X_test_all, columns=feat_names)
test_pdf['target'] = y_test_all

train_sdf = spark.createDataFrame(train_pdf).repartition(16)
test_sdf = spark.createDataFrame(test_pdf).repartition(4)

assembler = VectorAssembler(inputCols=feat_names, outputCol="features")
train_sdf = assembler.transform(train_sdf).cache()
test_sdf = assembler.transform(test_sdf).cache()

# Force materialization
train_count = train_sdf.count()
test_count = test_sdf.count()
load_time = time.time() - t0
print(f"Spark loaded: {train_count:,} train + {test_count:,} test in {load_time:.1f}s")

In [ ]:
# Train Tweedie GLM
t_train = time.time()

glr = GeneralizedLinearRegression(
    family="tweedie",
    variancePower=1.5,
    linkPower=0,
    featuresCol="features",
    labelCol="target",
    maxIter=50,
    regParam=0.01,
    tol=1e-6
)

spark_model = glr.fit(train_sdf)
time_spark_train = time.time() - t_train

# Predict
t_pred = time.time()
pred_sdf = spark_model.transform(test_sdf)
pred_pdf = pred_sdf.select("target", "prediction").toPandas()
pred_time_spark = time.time() - t_pred

time_spark = time.time() - t0
_, mem_spark = tracemalloc.get_traced_memory()
tracemalloc.stop()

y_true_sp = pred_pdf['target'].values
y_pred_sp = np.clip(pred_pdf['prediction'].values, 0, None)

mae_sp = mean_absolute_error(y_true_sp, y_pred_sp)
rmse_sp = np.sqrt(mean_squared_error(y_true_sp, y_pred_sp))
td_sp = mean_tweedie_deviance(y_true_sp, np.clip(y_pred_sp, 1e-6, None), power=1.5)
throughput_sp = len(y_true_sp) / pred_time_spark

benchmarks['pyspark'] = {
    'Framework': 'PySpark GLM',
    'Data Used': f'{split:,} (full, distributed)',
    'Train Time (s)': round(time_spark_train, 2),
    'Peak Memory (MB)': round(mem_spark / 1e6, 1),
    'Pred Time (s)': round(pred_time_spark, 2),
    'Throughput (rows/s)': f'{throughput_sp:,.0f}',
    'MAE': round(mae_sp, 2),
    'RMSE': round(rmse_sp, 2),
    'Tweedie Dev': round(td_sp, 4)
}

print(f"PySpark Complete!")
print(f"  Train: {time_spark_train:.1f}s | Total: {time_spark:.1f}s | Memory: {mem_spark/1e6:.1f} MB")
print(f"  MAE: {mae_sp:.2f} | RMSE: {rmse_sp:.2f} | TD: {td_sp:.4f}")
print(f"  Throughput: {throughput_sp:,.0f} rows/s")

# Print model summary
summary = spark_model.summary
print(f"  AIC: {summary.aic:.2f} | Deviance: {summary.deviance:.2f}")

spark.stop()

## 6. Dask: Out-of-Core Parallel Tweedie Regression

Dask partitions data across workers and parallelises both fitting (via subsampling) and prediction.

In [ ]:
import dask.array as da
from dask.distributed import Client, LocalCluster
from dask_ml.wrappers import ParallelPostFit

cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit='2GB')
client = Client(cluster)
print(f"Dask cluster: {len(client.scheduler_info()['workers'])} workers")

tracemalloc.start()
t0 = time.time()

# Scale features
scaler_dk = StandardScaler()
X_train_dk = scaler_dk.fit_transform(X_train_all)
X_test_dk = scaler_dk.transform(X_test_all)

# Fit on subsample (Dask excels at parallel prediction)
sub_size = 1_000_000
sub_idx = np.random.choice(split, sub_size, replace=False)

t_train = time.time()
dk_model = TweedieRegressor(power=1.5, alpha=0.1, link='log', max_iter=300)
dk_model.fit(X_train_dk[sub_idx], y_train_all[sub_idx])
time_dk_train = time.time() - t_train

# Wrap for parallel prediction
parallel = ParallelPostFit(dk_model)

# Create Dask array and predict in parallel
X_test_da = da.from_array(X_test_dk, chunks=(500_000, -1))

t_pred = time.time()
y_pred_dk = parallel.predict(X_test_da).compute()
pred_time_dk = time.time() - t_pred

time_dk = time.time() - t0
_, mem_dk = tracemalloc.get_traced_memory()
tracemalloc.stop()

y_pred_dk = np.clip(y_pred_dk, 0, None)
mae_dk = mean_absolute_error(y_test_all, y_pred_dk)
rmse_dk = np.sqrt(mean_squared_error(y_test_all, y_pred_dk))
td_dk = mean_tweedie_deviance(y_test_all, np.clip(y_pred_dk, 1e-6, None), power=1.5)
throughput_dk = len(y_test_all) / pred_time_dk

benchmarks['dask'] = {
    'Framework': 'Dask GLM',
    'Data Used': f'{sub_size:,} (train) / full (predict)',
    'Train Time (s)': round(time_dk_train, 2),
    'Peak Memory (MB)': round(mem_dk / 1e6, 1),
    'Pred Time (s)': round(pred_time_dk, 2),
    'Throughput (rows/s)': f'{throughput_dk:,.0f}',
    'MAE': round(mae_dk, 2),
    'RMSE': round(rmse_dk, 2),
    'Tweedie Dev': round(td_dk, 4)
}

print(f"Dask Complete!")
print(f"  Train: {time_dk_train:.1f}s | Total: {time_dk:.1f}s | Memory: {mem_dk/1e6:.1f} MB")
print(f"  MAE: {mae_dk:.2f} | RMSE: {rmse_dk:.2f} | TD: {td_dk:.4f}")
print(f"  Throughput: {throughput_dk:,.0f} rows/s")

client.close()
cluster.close()

## 7. Head-to-Head Comparison

In [ ]:
# Build comparison table
comp_df = pd.DataFrame(benchmarks).T
print("\n" + "="*100)
print("  LARGE-SCALE TWEEDIE FRAMEWORK COMPARISON (10M rows, 80% zeros)")
print("="*100)
print(comp_df.to_string())
print("="*100)

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

frameworks = ['Scikit-learn GLM', 'PyTorch NN', 'PySpark GLM', 'Dask GLM']
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

# Training Time
train_times = [benchmarks[k]['Train Time (s)'] for k in ['sklearn', 'pytorch', 'pyspark', 'dask']]
bars = axes[0, 0].bar(frameworks, train_times, color=colors, alpha=0.8, edgecolor='white')
axes[0, 0].set_ylabel('Training Time (seconds)')
axes[0, 0].set_title('Training Time Comparison')
for bar, val in zip(bars, train_times):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}s', ha='center', va='bottom', fontsize=10)
axes[0, 0].tick_params(axis='x', rotation=15)

# Peak Memory
mems = [benchmarks[k]['Peak Memory (MB)'] for k in ['sklearn', 'pytorch', 'pyspark', 'dask']]
bars = axes[0, 1].bar(frameworks, mems, color=colors, alpha=0.8, edgecolor='white')
axes[0, 1].set_ylabel('Peak Memory (MB)')
axes[0, 1].set_title('Memory Usage Comparison')
for bar, val in zip(bars, mems):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{val:.0f}', ha='center', va='bottom', fontsize=10)
axes[0, 1].tick_params(axis='x', rotation=15)

# Predictive Accuracy (MAE)
maes = [benchmarks[k]['MAE'] for k in ['sklearn', 'pytorch', 'pyspark', 'dask']]
bars = axes[1, 0].bar(frameworks, maes, color=colors, alpha=0.8, edgecolor='white')
axes[1, 0].set_ylabel('Mean Absolute Error')
axes[1, 0].set_title('Prediction Accuracy (MAE - lower is better)')
for bar, val in zip(bars, maes):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{val:.2f}', ha='center', va='bottom', fontsize=10)
axes[1, 0].tick_params(axis='x', rotation=15)

# Tweedie Deviance
tds = [benchmarks[k]['Tweedie Dev'] for k in ['sklearn', 'pytorch', 'pyspark', 'dask']]
bars = axes[1, 1].bar(frameworks, tds, color=colors, alpha=0.8, edgecolor='white')
axes[1, 1].set_ylabel('Tweedie Deviance')
axes[1, 1].set_title('Tweedie Deviance (lower is better)')
for bar, val in zip(bars, tds):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=10)
axes[1, 1].tick_params(axis='x', rotation=15)

plt.suptitle('Large-Scale Tweedie Framework Comparison\n(10M rows, 80% zeros, p=1.5)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('tweedie_benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PyTorch convergence plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(losses_pt)+1), losses_pt, 'o-', color='#FF5722', markersize=6, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Tweedie Loss', fontsize=12)
ax.set_title('PyTorch Training Convergence (8M rows)', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(1, len(losses_pt)+1))
plt.tight_layout()
plt.savefig('pytorch_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Prediction Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

models = {
    'Scikit-learn': y_pred_sk,
    'PyTorch': y_pred_pt,
    'PySpark': y_pred_sp,
    'Dask': y_pred_dk
}

for ax, (name, preds) in zip(axes.flat, models.items()):
    # Plot actual vs predicted distributions
    ax.hist(y_test_all[y_test_all > 0], bins=100, alpha=0.5, color='gray',
            label='Actual (non-zero)', density=True)
    ax.hist(preds[preds > 1], bins=100, alpha=0.5, color='steelblue',
            label='Predicted (> $1)', density=True)
    ax.set_xlabel('Value ($)')
    ax.set_ylabel('Density')
    ax.set_title(f'{name} - Prediction Distribution')
    ax.legend(fontsize=9)
    ax.set_xlim(0, np.percentile(y_test_all[y_test_all>0], 95))

plt.suptitle('Predicted vs Actual Distributions (Non-Zero Values)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('prediction_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Conclusions and Recommendations

### Key Findings

| Framework | Best For | Watch Out For |
|-----------|----------|---------------|
| **Scikit-learn** | Quick prototyping, small-medium data (<1M rows) | Cannot scale to large datasets |
| **PyTorch** | Complex non-linear patterns, GPU acceleration, custom architectures | Requires more code, hyperparameter tuning |
| **PySpark** | Massive datasets (>100M rows), cluster deployment, SQL pipelines | Overhead for small data, limited model types |
| **Dask** | Out-of-core on single machine, scikit-learn API compatibility | Training still uses subsampling, limited native Tweedie distribution support |

### When to Use Which

1. **< 1M rows**: Use scikit-learn `TweedieRegressor` or XGBoost/LightGBM with Tweedie objective
2. **1M - 50M rows**: PyTorch with mini-batch training (especially with GPU) or Dask for parallel prediction
3. **> 50M rows**: PySpark on a cluster for true distributed training, or PyTorch with distributed data parallel
4. **High-cardinality categoricals**: PyTorch with embedding layers

### The Tweedie Advantage

Across all frameworks and scales, the Tweedie distribution provides:
- **Native zero handling**: No need for separate frequency/severity models
- **No transformation bias**: Direct modeling of the raw target
- **Statistical foundation**: Grounded in exponential dispersion family theory
- **Framework agnostic**: Works as a drop-in loss function in any regression framework